# 문제 12. 항공지원 상황실의 운항 기록 정제와 거점별 실적 집계

**제한시간 30분 · 난이도 최상 목표 · Python3 · Jupyter · CPU · CSV 제출**

## 상황

항공지원 상황실에서 운항 계획과 여러 차례 수신된 결과 보고를 결합해 일일 현황표를 작성하려고 합니다. 지문에 정의된 기준으로 거점별 운항 실적과 최대 동시 운항 수를 계산하세요. **예측 모델 학습은 필요하지 않습니다.**

실제 공개 민간 항공편 기록을 가상 재난지원 항공운용 상황으로 재구성했습니다. 군·무인기 운항 실측값은 아닙니다. ID 익명화, 보고 분할·중복 전송, 일부 필드만 있는 초기 보고는 출제용 가공입니다. 최신 보고의 운항 값은 실제 원본 값이며, 거점 규칙과 실적 없는 참조 거점은 연습용 설정입니다.

## Data

| 파일 | 내용 |
|---|---|
| data/flight_plan.csv | 운항 계획32,731건. flight_id, base_id, flight_date, scheduled_departure_hhmm |
| data/reports_a.csv, reports_b.csv | 합계44,187행. 분할·중복·수정 버전이 섞인 결과 보고 |
| data/base_rules.csv | 거점별 운영일 시작 시각(day_start_hour), 지연 기준(delay_limit_min) |
| data/targets.csv | 집계할49개 operation_date·base_id 조합과 row_id. 이 순서로 제출 |
| data/sample_submission.csv | 제출 컬럼·행 순서 예시 |

### 결과 보고 컬럼

| 컬럼 | 의미 |
|---|---|
| flight_id, revision | 운항 식별자, 보고 버전 번호 |
| actual_departure_hhmm, actual_arrival_hhmm | 실제 출발·도착지 현지 시각(HHMM), 결측 가능 |
| departure_delay_min | 예정 출발 대비 지연 분. 조기 출발이면 음수 |
| cancelled, diverted | 취소·회항 여부(1/0) |
| elapsed_min | 실제 출발부터 도착까지 경과 분. 지상 이동을 포함 |
| airborne_min | 공중 비행 분. 결측 가능 |

`flight_date`와 예정 출발 시각은 출발지 현지 기준입니다. 도착지 현지 시각을 출발지 시각에서 직접 빼면 시간대 차이 때문에 잘못된 결과가 나올 수 있습니다.

## 초기 코드


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

BASE_DIR = Path(".")
plan = pd.read_csv(BASE_DIR / "data/flight_plan.csv")
reports_a = pd.read_csv(BASE_DIR / "data/reports_a.csv")
reports_b = pd.read_csv(BASE_DIR / "data/reports_b.csv")
base_rules = pd.read_csv(BASE_DIR / "data/base_rules.csv")
targets = pd.read_csv(BASE_DIR / "data/targets.csv")
sample_submission = pd.read_csv(BASE_DIR / "data/sample_submission.csv")
display(plan.head())
display(reports_a.head())


## 과제

### 1. 보고 통합

두 보고 파일을 합쳐 **flight_id별 revision이 가장 큰 보고 한 건**을 사용하세요. 동일 ID·동일 버전의 중복은 값이 같으므로 하나만 남깁니다. 파일 순서나 마지막 행이 최신이라는 보장은 없습니다. 초기·최신 보고를 열별로 섞어서 보완하지 마세요.

계획을 기준으로 보고·거점 규칙을 결합합니다. 결측 행을 일괄 삭제하지 마세요. 파일에 있는 전체 운항을 처리한 뒤 지표에 맞게 집계 대상을 정하세요.

### 2. 예정 시각과 운영일

- `scheduled_dt = flight_date + scheduled_departure_hhmm의 시각`.
- HHMM은 십진 시간이나 분이 아닙니다. 530은05:30, 0은00:00입니다. 2400이 있으면 다음 날00:00으로 해석합니다.
- **계획 운영일**은 `scheduled_dt - day_start_hour시간`의 날짜입니다.
- 예: 시작 시각이06:00이면1월11일05:59 예정편은1월10일 운영일,1월11일06:00 예정편은1월11일 운영일입니다.

### 3. 상태 분류 — 아래 순서대로 한 상태만 적용

1. cancelled=1 → **취소**. 다른 값과 관계없이 우선합니다.
2. 취소가 아니고 diverted=1 → **회항**.
3. cancelled=0, diverted=0이고 실제 출발·도착 시각과 departure_delay_min이 모두 존재하며 elapsed_min>0 → **완료**.
4. 그 외 → **미확정**.

### 4. 제출할 지표

아래 첫9개 지표는 **거점·계획 운영일**로 묶어 계산합니다.

| 컬럼 | 계산 규칙 |
|---|---|
| planned_count | 해당 그룹의 계획 운항 수 |
| cancelled_count | 취소 수 |
| diverted_count | 회항 수 |
| completed_count | 완료 수 |
| unknown_count | 미확정 수 |
| cancel_rate | 취소 수 / 계획 수. 계획0건이면-1 |
| delay_rate | 완료 중 departure_delay_min≥delay_limit_min인 수 / 완료 수. 완료0건이면-1 |
| mean_delay_min | 완료편의 max(departure_delay_min,0) 평균. 완료0건이면-1 |
| airborne_hours | 완료편 중 airborne_min이 존재하고0 이상인 값의 합 /60. 해당 값이 없으면0 |

마지막 **peak_active**는 계획 운영일 집계와 다르게, **실제 운항 구간과 대상 운영일의 겹침**을 계산합니다.

- 완료편만 사용합니다. 시작=`scheduled_dt + departure_delay_min분`, 종료=`시작 + elapsed_min분`입니다. 이 구간은 공중 체류만을 뜻하지 않으며 지상 이동을 포함합니다.
- 대상 운영일 구간은 그 날짜의 day_start_hour부터 다음 날 같은 시각까지입니다.
- 운항 구간과 운영일 모두 **시작 포함·종료 제외 `[시작,종료)`**입니다.
- 그 구간 중 동시에 진행 중인 운항 수의 최댓값이 peak_active입니다. 같은 시각에 종료·시작하면 종료편을 먼저 제외합니다.
- 계획 운영일이 전날이어도 실제 구간이 겹치면 포함합니다. **targets에 없는 계획 운영일의 운항도 먼저 삭제하면 안 됩니다.** 전체 제공 기록 중 완료편을 대상으로 계산하세요.
- 운항이 전혀 없으면0입니다. 시간대가 다른 거점끼리는 합치지 않습니다.

시간 규칙 확인용 예: 같은 거점에서 A가05:50~06:10, B가06:10~06:20 운항했다면06:00 시작 운영일의 peak_active는1입니다. C가06:05~06:15 추가되면2입니다.

### 5. 출력과 평가

Notebook 옆에 **submission.csv**를 저장하세요. targets.csv의 row_id 순서를 그대로 사용하고, 실적 없는 거점·날짜도 포함하세요.

컬럼 순서:

`row_id,planned_count,cancelled_count,diverted_count,completed_count,unknown_count,cancel_rate,delay_rate,mean_delay_min,airborne_hours,peak_active`

- 49행. 개수 지표는0 이상의 정수, 비율은0~1 또는 지정된-1입니다.
- 실수 지표는 소수점6자리로 반올림하세요. 결측·무한대·퍼센트 문자열은 금지합니다.
- 개수는 정확히 일치, 실수는 절대 오차1e-6 이내면 정답입니다.
- **지표별 정답 셀 비율×아래 배점**을 합산합니다. 계획·취소·회항·미확정 수는각5점, 완료 수는10점, 취소율·지연율·평균 지연·비행시간은각10점, **peak_active는30점**으로 총100점입니다. 계산 문제이므로 완전한 정답은100점입니다. 공식 대회 배점·합격선은 아닙니다.
- `index=False`로 저장하고 코드·실행 결과도 Notebook에 저장하세요. 별도 보고서는 없습니다.


In [ ]:
# 여기에 코드를 작성하세요.


## CSV 저장 예시

In [ ]:
# result: 위에서 계산한 집계 결과 DataFrame
# result.to_csv(BASE_DIR / "submission.csv", index=False, encoding="utf-8-sig")


## 파일 형식 확인

In [ ]:
path = BASE_DIR / "submission.csv"
if path.exists():
    result = pd.read_csv(path)
    assert result.columns.tolist() == sample_submission.columns.tolist()
    assert result.row_id.tolist() == targets.row_id.tolist()
    values = result.iloc[:, 1:].to_numpy(dtype=float)
    assert np.isfinite(values).all()
    for c in ["planned_count","cancelled_count","diverted_count","completed_count","unknown_count","peak_active"]:
        assert (result[c] >= 0).all() and (result[c] == np.floor(result[c])).all()
    print("제출 형식 확인 완료. 점수 채점이나 최종 제출은 아닙니다.")
else:
    print("풀이 후 submission.csv를 생성하세요.")
